# GLDM Delta Rollover — TLT Hedge (Live)

**Strategy.** Hold a long position in **GLDM** (SPDR Gold MiniShares Trust) and hedge the rate-sensitive component of the gold exposure with a short position in **TLT** (iShares 20+ Year Treasury Bond ETF). Re-estimate the hedge ratio on a rolling window ("delta") and roll the hedge on a schedule or when drift exceeds a threshold.

**Data Source:** Alpha Vantage (live) — requires API key  
**Entry Point:** `GLDM_Delta_Rollover_TLT_Hedge_Live(api_key)`

---

| Section | Purpose |
|---|---|
| **§1 — Environment & Configuration** | Imports, dataclass config, logging |
| **§2 — Alpha Vantage Data Client** | Live daily-adjusted OHLCV with retry & caching |
| **§3 — Hedge Ratio & Delta Engine** | Rolling OLS beta, hedge notionals, roll signal |
| **§4 — Live Report** | Position table, roll recommendation, CSV export |
| **§5 — Entry Point** | `GLDM_Delta_Rollover_TLT_Hedge_Live(api_key)` |

---
## 1. Environment & Configuration

In [ ]:
# =============================================================================
# §1.1 — Imports
# =============================================================================
import os
import sys
import json
import time
import hashlib
import logging
import warnings
from datetime import datetime, timedelta
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Tuple, Dict

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings('ignore', category=FutureWarning)
logging.getLogger('urllib3').setLevel(logging.CRITICAL)

print(f"python:   {sys.version.split()[0]}")
print(f"pandas:   {pd.__version__}")
print(f"numpy:    {np.__version__}")
print(f"requests: {requests.__version__}")

In [ ]:
# =============================================================================
# §1.2 — Configuration Dataclass
# =============================================================================
@dataclass
class DeltaHedgeConfig:
    """Central config — no magic numbers in logic cells."""

    # Universe
    asset_ticker: str = 'GLDM'
    hedge_ticker: str = 'TLT'

    # Alpha Vantage
    alpha_vantage_key: str = ''
    av_base_url: str = 'https://www.alphavantage.co/query'
    av_output_size: str = 'full'
    request_timeout: int = 20
    request_spacing_sec: float = 0.8
    max_retries: int = 4
    retry_delay: float = 2.0

    # Cache
    cache_dir: str = './av_cache'
    cache_ttl_hours: int = 12
    use_cache: bool = True

    # Hedge engine
    beta_window: int = 60
    min_bars: int = 252
    roll_threshold_bps: float = 150.0
    roll_calendar_days: int = 21
    target_asset_notional: float = 1_000_000.0

    # Output
    output_dir: str = './delta_hedge_output'

---
## 2. Alpha Vantage Data Client

Live daily-adjusted OHLCV pulls using Alpha Vantage's `TIME_SERIES_DAILY_ADJUSTED` endpoint (Premium). Falls back to `TIME_SERIES_DAILY` if the adjusted endpoint is unavailable. Caches parsed DataFrames as parquet with a TTL.

In [ ]:
# =============================================================================
# §2.1 — Cache Manager
# =============================================================================
class CacheManager:
    """File-based cache (parquet + JSON metadata) with TTL."""

    def __init__(self, config: DeltaHedgeConfig):
        self.enabled = config.use_cache
        self.cache_dir = Path(config.cache_dir)
        self.ttl = timedelta(hours=config.cache_ttl_hours)
        if self.enabled:
            self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _paths(self, key: str) -> Tuple[Path, Path]:
        h = hashlib.md5(key.encode()).hexdigest()
        return self.cache_dir / f'{h}.parquet', self.cache_dir / f'{h}.json'

    def get(self, key: str) -> Optional[pd.DataFrame]:
        if not self.enabled:
            return None
        data_path, meta_path = self._paths(key)
        if not (data_path.exists() and meta_path.exists()):
            return None
        try:
            meta = json.loads(meta_path.read_text())
            stamped = datetime.fromisoformat(meta['stamped_at'])
            if datetime.now() - stamped > self.ttl:
                return None
            return pd.read_parquet(data_path)
        except Exception:
            return None

    def put(self, key: str, df: pd.DataFrame) -> None:
        if not self.enabled or df is None or df.empty:
            return
        data_path, meta_path = self._paths(key)
        try:
            df.to_parquet(data_path)
            meta_path.write_text(json.dumps({
                'key': key,
                'stamped_at': datetime.now().isoformat(),
                'rows': len(df),
            }))
        except Exception as e:
            print(f'  [WARN] cache write failed for {key}: {e}')

In [ ]:
# =============================================================================
# §2.2 — Alpha Vantage Client
# =============================================================================
class AlphaVantageClient:
    """Minimal Alpha Vantage client for daily OHLCV.

    Replaces yfinance. Prefers TIME_SERIES_DAILY_ADJUSTED (Premium) and
    falls back to TIME_SERIES_DAILY if the adjusted feed is gated.
    """

    def __init__(self, config: DeltaHedgeConfig, cache: CacheManager):
        if not config.alpha_vantage_key:
            raise ValueError('Alpha Vantage API key is required.')
        self.config = config
        self.cache = cache
        self._last_request_ts: float = 0.0

    def _throttle(self) -> None:
        delta = time.time() - self._last_request_ts
        if delta < self.config.request_spacing_sec:
            time.sleep(self.config.request_spacing_sec - delta)
        self._last_request_ts = time.time()

    def _request(self, params: Dict[str, str]) -> Dict:
        params = dict(params)
        params['apikey'] = self.config.alpha_vantage_key
        last_err: Optional[str] = None
        for attempt in range(self.config.max_retries):
            self._throttle()
            try:
                resp = requests.get(
                    self.config.av_base_url,
                    params=params,
                    timeout=self.config.request_timeout,
                )
                resp.raise_for_status()
                payload = resp.json()
                if 'Note' in payload or 'Information' in payload:
                    last_err = payload.get('Note') or payload.get('Information')
                    time.sleep(self.config.retry_delay * (2 ** attempt))
                    continue
                if 'Error Message' in payload:
                    raise RuntimeError(payload['Error Message'])
                return payload
            except (requests.RequestException, ValueError) as e:
                last_err = str(e)
                time.sleep(self.config.retry_delay * (2 ** attempt))
        raise RuntimeError(f'Alpha Vantage request failed: {last_err}')

    @staticmethod
    def _parse_time_series(payload: Dict) -> pd.DataFrame:
        series_key = next(
            (k for k in payload.keys() if k.startswith('Time Series')),
            None,
        )
        if series_key is None:
            raise RuntimeError(f'No time series in payload (keys={list(payload.keys())})')

        rows = []
        for date_str, bar in payload[series_key].items():
            close = bar.get('5. adjusted close') or bar.get('4. close')
            rows.append({
                'Date': date_str,
                'Open': float(bar.get('1. open', 'nan')),
                'High': float(bar.get('2. high', 'nan')),
                'Low': float(bar.get('3. low', 'nan')),
                'Close': float(bar.get('4. close', 'nan')),
                'AdjClose': float(close) if close is not None else float('nan'),
                'Volume': float(bar.get('6. volume') or bar.get('5. volume') or 0),
            })
        df = pd.DataFrame(rows)
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.set_index('Date').sort_index()
        return df

    def get_daily(self, symbol: str) -> pd.DataFrame:
        cache_key = f'av_daily::{symbol}::{self.config.av_output_size}'
        cached = self.cache.get(cache_key)
        if cached is not None:
            return cached

        for function in ('TIME_SERIES_DAILY_ADJUSTED', 'TIME_SERIES_DAILY'):
            try:
                payload = self._request({
                    'function': function,
                    'symbol': symbol,
                    'outputsize': self.config.av_output_size,
                    'datatype': 'json',
                })
                df = self._parse_time_series(payload)
                if df.empty:
                    continue
                self.cache.put(cache_key, df)
                return df
            except RuntimeError as e:
                print(f'  [WARN] {function} failed for {symbol}: {e}')
                continue
        raise RuntimeError(f'No Alpha Vantage data returned for {symbol}')

    def get_many(self, symbols) -> Dict[str, pd.DataFrame]:
        out: Dict[str, pd.DataFrame] = {}
        for sym in symbols:
            print(f'  [AV] downloading {sym} …')
            out[sym] = self.get_daily(sym)
        return out

---
## 3. Hedge Ratio & Delta Engine

Rolling OLS regression of **GLDM** daily log returns on **TLT** daily log returns over `beta_window` trading days. The fitted slope is the **delta** — the dollar sensitivity of GLDM to a unit move in TLT. The TLT hedge notional is

$$\text{TLT}_{\text{notional}} = -\beta_t \cdot \frac{P^{\text{GLDM}}_t}{P^{\text{TLT}}_t} \cdot N^{\text{GLDM}}$$

A **roll signal** fires when the realised hedge drift exceeds `roll_threshold_bps` or the calendar-day age exceeds `roll_calendar_days`.

In [ ]:
# =============================================================================
# §3.1 — Price Frame Assembly
# =============================================================================
def build_price_frame(prices: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Join adjusted closes for the asset and hedge onto a shared index."""
    frames = []
    for sym, df in prices.items():
        px = df['AdjClose'].rename(sym) if 'AdjClose' in df.columns else df['Close'].rename(sym)
        frames.append(px)
    out = pd.concat(frames, axis=1).dropna(how='any')
    return out

In [ ]:
# =============================================================================
# §3.2 — Rolling Beta (Delta)
# =============================================================================
def rolling_beta(returns: pd.DataFrame, asset: str, hedge: str, window: int) -> pd.Series:
    """Rolling OLS slope of asset on hedge via cov/var."""
    cov = returns[asset].rolling(window).cov(returns[hedge])
    var = returns[hedge].rolling(window).var()
    return (cov / var).rename('beta')

def compute_delta_hedge(prices: pd.DataFrame, config: DeltaHedgeConfig) -> pd.DataFrame:
    """Compute the live delta, hedge notional, and roll flags."""
    asset, hedge = config.asset_ticker, config.hedge_ticker
    if len(prices) < config.min_bars:
        raise RuntimeError(
            f'Insufficient history: {len(prices)} rows < min_bars={config.min_bars}'
        )

    log_returns = np.log(prices).diff()
    beta = rolling_beta(log_returns, asset, hedge, config.beta_window)

    hedge_units = -(beta * prices[asset] / prices[hedge]) * (
        config.target_asset_notional / prices[asset]
    )
    hedge_notional = hedge_units * prices[hedge]

    table = pd.DataFrame({
        f'{asset}_price': prices[asset],
        f'{hedge}_price': prices[hedge],
        'beta': beta,
        f'{asset}_units': config.target_asset_notional / prices[asset],
        f'{hedge}_units': hedge_units,
        f'{hedge}_notional': hedge_notional,
    }).dropna()

    table['beta_drift_bps'] = (table['beta'].diff().abs() * 10_000).fillna(0.0)
    table['bars_since_roll'] = np.nan
    last_roll_idx = table.index[0]
    for idx in table.index:
        days = (idx - last_roll_idx).days
        table.loc[idx, 'bars_since_roll'] = days
        drift_trigger = table.loc[idx, 'beta_drift_bps'] > config.roll_threshold_bps
        calendar_trigger = days >= config.roll_calendar_days
        if drift_trigger or calendar_trigger:
            last_roll_idx = idx
            table.loc[idx, 'roll_signal'] = 1
        else:
            table.loc[idx, 'roll_signal'] = 0
    table['roll_signal'] = table['roll_signal'].astype(int)
    return table

---
## 4. Live Report

In [ ]:
# =============================================================================
# §4.1 — Status Banner
# =============================================================================
def print_status(config: DeltaHedgeConfig, hedge_tbl: pd.DataFrame) -> None:
    latest = hedge_tbl.iloc[-1]
    asof = hedge_tbl.index[-1].strftime('%Y-%m-%d')
    asset, hedge = config.asset_ticker, config.hedge_ticker
    print('=' * 72)
    print(f'  GLDM DELTA ROLLOVER — TLT HEDGE (LIVE)')
    print(f'  As of {asof}   (generated {datetime.now().strftime("%Y-%m-%d %H:%M:%S")})')
    print('=' * 72)
    print(f'  {asset} price:        ${latest[f"{asset}_price"]:>12,.2f}')
    print(f'  {hedge} price:        ${latest[f"{hedge}_price"]:>12,.2f}')
    print(f'  Rolling beta ({config.beta_window}d):  {latest["beta"]:>12,.4f}')
    print(f'  Target {asset} notional: ${config.target_asset_notional:>12,.0f}')
    print(f'  {asset} units (long):  {latest[f"{asset}_units"]:>12,.2f}')
    print(f'  {hedge} units (hedge): {latest[f"{hedge}_units"]:>12,.2f}')
    print(f'  {hedge} notional:     ${latest[f"{hedge}_notional"]:>12,.0f}')
    print(f'  Beta drift (bps):     {latest["beta_drift_bps"]:>12,.1f}')
    print(f'  Roll signal today:    {"YES" if latest["roll_signal"] == 1 else "no"}')
    print('=' * 72)

In [ ]:
# =============================================================================
# §4.2 — CSV Export
# =============================================================================
def export_hedge_report(hedge_tbl: pd.DataFrame, config: DeltaHedgeConfig) -> Path:
    out_dir = Path(config.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = out_dir / f'GLDM_TLT_delta_hedge_{stamp}.csv'
    hedge_tbl.to_csv(path, index=True)
    latest_path = out_dir / 'GLDM_TLT_delta_hedge_latest.csv'
    hedge_tbl.to_csv(latest_path, index=True)
    print(f'  [OK] wrote {path}')
    print(f'  [OK] wrote {latest_path}')
    return path

---
## 5. Entry Point

Call `GLDM_Delta_Rollover_TLT_Hedge_Live(api_key)` with your Alpha Vantage key. Returns the latest hedge row as a `pd.Series` and writes a CSV to `./delta_hedge_output/`.

In [ ]:
# =============================================================================
# §5.1 — Main Entry Point
# =============================================================================
def GLDM_Delta_Rollover_TLT_Hedge_Live(
    api_key: str,
    config: Optional[DeltaHedgeConfig] = None,
) -> pd.Series:
    """Fetch GLDM and TLT live from Alpha Vantage and compute the delta-rollover hedge.

    Parameters
    ----------
    api_key : str
        Alpha Vantage API key (Premium recommended for adjusted series).
    config : DeltaHedgeConfig, optional
        Override default configuration.

    Returns
    -------
    pd.Series
        Most recent hedge row (price, beta, units, notional, roll_signal).
    """
    cfg = config or DeltaHedgeConfig()
    cfg.alpha_vantage_key = api_key

    cache = CacheManager(cfg)
    client = AlphaVantageClient(cfg, cache)

    prices_raw = client.get_many([cfg.asset_ticker, cfg.hedge_ticker])
    prices = build_price_frame(prices_raw)

    hedge_tbl = compute_delta_hedge(prices, cfg)
    print_status(cfg, hedge_tbl)
    export_hedge_report(hedge_tbl, cfg)
    return hedge_tbl.iloc[-1]

In [ ]:
# =============================================================================
# §5.2 — Run Live
# =============================================================================
latest = GLDM_Delta_Rollover_TLT_Hedge_Live('A9R0FMDJEHEDNWAZ')
latest